# Learning Python libraries for JSON-LD

Trying out two libraries:

* [RDFLib/rdflib-jsonld: JSON-LD parser and serializer plugins for RDFLib (Python 2.6+)](https://github.com/RDFLib/rdflib-jsonld)
* [digitalbazaar/pyld: JSON-LD processor written in Python](https://github.com/digitalbazaar/pyld)

In [ ]:
from rdflib import Graph, plugin, ConjunctiveGraph
from rdflib.serializer import Serializer

from pyld import jsonld
import json

import requests

In [ ]:
testrdf = '''
    @prefix dc: <http://purl.org/dc/terms/> .
    <http://example.org/about>
    dc:title "Someone's Homepage"@en .
'''

g = Graph().parse(data=testrdf, format='n3')
print(g.serialize(format='json-ld', indent=4).decode('utf-8'))

[rdflib-jsonld/test_api.py at cc5f005b222105724cd59c6069df9982fbd28c98 · RDFLib/rdflib-jsonld](https://github.com/RDFLib/rdflib-jsonld/blob/cc5f005b222105724cd59c6069df9982fbd28c98/test/test_api.py#L7)

In [ ]:
# using rdflib-jsonld to parse JSON-LD into Graph

from rdflib.plugin import register, Parser, Serializer
register('json-ld', Parser, 'rdflib_jsonld.parser', 'JsonLDParser')
register('json-ld', Serializer, 'rdflib_jsonld.serializer', 'JsonLDSerializer')

from rdflib import Graph, Literal, URIRef


def test_parse():  
    test_json = '''
    {
        "@context": {
            "dc": "http://purl.org/dc/terms/",
            "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
            "rdfs": "http://www.w3.org/2000/01/rdf-schema#"
        },
        "@id": "http://example.org/about",
        "dc:title": {
            "@language": "en",
            "@value": "Someone's Homepage"
        }
    }
    '''
    g = Graph().parse(data=test_json, format='json-ld')
    assert list(g) == [(
        URIRef('http://example.org/about'),
        URIRef('http://purl.org/dc/terms/title'),
        Literal("Someone's Homepage", lang='en'))]
    
test_parse()

[digitalbazaar/pyld: JSON-LD processor written in Python](https://github.com/digitalbazaar/pyld)

In [ ]:
# compact, expanded, flattened, framed display of jsonld

doc = {
    "http://schema.org/name": "Manu Sporny",
    "http://schema.org/url": {"@id": "http://manu.sporny.org/"},
    "http://schema.org/image": {"@id": "http://manu.sporny.org/images/manu.png"}
}

context = {
    "name": "http://schema.org/name",
    "homepage": {"@id": "http://schema.org/url", "@type": "@id"},
    "image": {"@id": "http://schema.org/image", "@type": "@id"}
}

# compact a document according to a particular context
# see: http://json-ld.org/spec/latest/json-ld/#compacted-document-form
compacted = jsonld.compact(doc, context)

print(json.dumps(compacted, indent=2))

In [ ]:
# expand a document, removing its context
# see: http://json-ld.org/spec/latest/json-ld/#expanded-document-form
expanded = jsonld.expand(compacted)
print(json.dumps(expanded, indent=2))

In [ ]:
# flatten a document
# see: http://json-ld.org/spec/latest/json-ld/#flattened-document-form
flattened = jsonld.flatten(doc)
# all deep-level trees flattened to the top-level
flattened

In [ ]:
# frame a document
# see: http://json-ld.org/spec/latest/json-ld-framing/#introduction
# framed = jsonld.frame(doc, frame)
# document transformed into a particular tree structure per the given frame

In [ ]:
# normalize a document using the RDF Dataset Normalization Algorithm
# (URDNA2015), see: http://json-ld.github.io/normalization/spec/
normalized = jsonld.normalize(
    doc, {'algorithm': 'URDNA2015', 'format': 'application/n-quads'})
# normalized is a string that is a canonical representation of the document
# that can be used for hashing, comparison, etc.

normalized

# DINAA


[View of Digital Index of North American Archaeology (DINAA)](https://opencontext.org/projects/416A274C-CF88-4471-3E31-93DB825E9E4A)

https://opencontext.org/projects/416A274C-CF88-4471-3E31-93DB825E9E4A.jsonld


In [ ]:
url = "https://opencontext.org/projects/416A274C-CF88-4471-3E31-93DB825E9E4A.jsonld"
r = requests.get(url)
dinaa_json = r.json()

dinaa_json

In [ ]:
g = Graph().parse(data=json.dumps(dinaa_json), format='json-ld')

In [ ]:
from collections import Counter

c = Counter()

for s,p,o in g:
    c.update([p])
    
c

In [ ]:
r = jsonld.load_document(url)
r

In [ ]:
jsonld.expand(url)

In [ ]:
def triples_from_url(url):
    # r = requests.get(url)
    
    g = Graph().parse(format='json-ld', location=url)
    return list(g)
    
def count_predicates(triples):
    
    c = Counter()
    for (s, p, o) in triples:
        c.update([p])
    
    return c

In [ ]:
triples = triples_from_url("https://opencontext.org/projects/3FAAA477-5572-4B05-8DC1-CA264FE1FC10.jsonld")

In [ ]:
c = count_predicates(triples)
c.most_common()

In [ ]:
# let's look at the subjects
# way of loading graph with a set "base" of some sort...I'm getting references to 
# this Jupyter notebook

s_c = Counter()
o_c = Counter()
p_c = Counter()


for (s,p,o) in triples:
    s_c.update([type(s)])
    o_c.update([type(o)])
    p_c.update([type(p)])

    
    
s_c,p_c, o_c

# Drawing d-ss-002 (Image) 

https://opencontext.org/media/37861659-67ac-4864-b93a-0300411e6b1e#tab_obs-1

In [ ]:
url = "https://opencontext.org/media/37861659-67ac-4864-b93a-0300411e6b1e.jsonld"
#triples = triples_from_url(url)

In [ ]:
g = ConjunctiveGraph()
g.parse(format='json-ld', location=url)

g

In [ ]:
type(g)

In [ ]:
print(g.serialize(format='nquads').decode('utf-8'))

In [ ]:
# interesting thing is to list out all the predicates from an OpenContext document to see the data structures

In [ ]:
# sqarql queries on graph

# http://opencontext.org/vocabularies/oc-general/slug
    
qres = g.query(
    """SELECT ?s  ?o WHERE {
          ?s <http://opencontext.org/vocabularies/oc-general/slug> ?o.
       }""")

list(qres)

In [ ]:
qres = g.query(
    """SELECT ?s  ?o WHERE {
          ?s oc-gen:slug ?o.
       }""",
    initNs = { "oc-gen": URIRef('http://opencontext.org/vocabularies/oc-general/')}
)


len(qres)


In [ ]:
# understand the @context 

url = "https://opencontext.org/media/37861659-67ac-4864-b93a-0300411e6b1e.jsonld"

jsonld.expand(url)

In [ ]:
# https://github.com/digitalbazaar/pyld/blob/master/lib/pyld/jsonld.py#L1163
list(jsonld._cache.get('activeCtx').cache.keys())

In [ ]:
# study the subjects and objects too --> especially(?) declared types

In [ ]:
# parse out the external contexts that are in many OpenContext jsonld

# field drawing thumbnails for project 

In [ ]:
url = "https://opencontext.org/search.jsonld/?prop=101-image-type---101-field-drawing&proj=101-arce-sphinx-project-1979-1983-archive#18/29.97499/31.13762/20/any/Google-Satellite"
g = ConjunctiveGraph()
g.parse(format='json-ld', location=url)

In [ ]:
len(g)

In [ ]:
c = Counter()
for (s,p,o) in g:
    c.update([p])
    if 'properties' in p.toPython():
        print (s,o)

c

In [ ]:
p.toPython()

In [ ]:
'dog' in 'doge'

In [ ]:
qres = g.query(
    """SELECT ?p ?o WHERE {
          <https://opencontext.org/search.jsonld/?prop=101-image-type---101-field-drawing&proj=101-arce-sphinx-project-1979-1983-archive#record-8-of-204> 
          ?p ?o.
       }""",
    initNs = { "oc-gen": URIRef('http://opencontext.org/vocabularies/oc-general/')}
)


```
{
"id": "#record-8-of-204",
"label": "Drawing d-ne-008",
"rdfs:isDefinedBy": "http://opencontext.org/media/7e8ca0a8-cdf7-48ff-a401-e67489037c69",
"type": "Feature",
"category": "oc-api:geo-record",
"geometry": {
"id": "#geo-rec-geom-8-of-204",
"type": "Point",
"coordinates": [
31.13792855839629,
29.975383029433967
]
},
"properties": {
"id": "#rec-8-of-204",
"feature-type": "item record",
"uri": "http://opencontext.org/media/7e8ca0a8-cdf7-48ff-a401-e67489037c69",
"href": "https://opencontext.org/media/7e8ca0a8-cdf7-48ff-a401-e67489037c69",
"citation uri": false,
"label": "Drawing d-ne-008",
"project label": "ARCE Sphinx Project 1979-1983 Archive",
"project href": "https://opencontext.org/projects/141e814a-ba2d-4560-879f-80f1afb019e9",
"context label": "Egypt/Giza/Sphinx Amphitheater/Sphinx Ditch/Sphinx Northeast/Feature FNEh5",
"context href": "https://opencontext.org/subjects/e975b2bc-0184-430c-8543-ef9369e3fd8e",
"early bce/ce": false,
"late bce/ce": false,
"item category": "Image",
"thumbnail": "https://artiraq.org/static/opencontext/giza-sphinx/thumbs/Drawings/101-drawing-d-ne-008.jpg",
"published": "2017-11-14T20:44:20Z",
"updated": "2018-03-23T01:06:18Z",
"Image Type": [
"Field Drawing"
]
}
},
```

In [ ]:
list(qres)

In [ ]:
# https://opencontext.org/search.jsonld/?prop=101-image-type---101-field-drawing&proj=101-arce-sphinx-project-1979-1983-archive#rec-8-of-204


qres = g.query(
    """SELECT ?p ?o WHERE {
          <https://opencontext.org/search.jsonld/?prop=101-image-type---101-field-drawing&proj=101-arce-sphinx-project-1979-1983-archive#rec-8-of-204> 
          ?p ?o.
       }"""
)

list(qres)

ok -- I don't think parsing the output as jsonld doesn't let us get at the thumbnails.

In [ ]:
url = "https://opencontext.org/search.jsonld/?prop=101-image-type---101-field-drawing&proj=101-arce-sphinx-project-1979-1983-archive#18/29.97499/31.13762/20/any/Google-Satellite"
r = requests.get(url)

In [ ]:
r.json().get('next')

In [ ]:
def thumbnails_for_url(url):
    more_records = True
    if more_records:
        r = requests.get(url)

        for f in r.json()['features']:
            thumbnail = f.get('properties', {}).get('thumbnail')
            if thumbnail is not None:
                yield thumbnail

        next_link = r.json().get('next')
        if not next_link:
            more_records = False
            

In [ ]:
from itertools import islice

In [ ]:
list(islice(thumbnails_for_url(url),30))